# End-to-end machine learning workflows with HPE Ezmeral ML Ops - Lab 3
## Data pre-processing, model development and model training

### About the Model

The goal of this machine learning project is to classify whether a person makes over $50K a year or not given their demographic variation. This is what we refer to as _prediction_. To achieve this project, several classification techniques based on XGBoost (Extreme Gradient Boosting) are experimented with a variety of parameters to determine the model that yields to the best prediction result.

This tutorial uses a XGboost based python model written to classify a person's income as being either less than or equal to, or more than, $ 50,000.

This tutorial generates a classification prediction of individuals based on the US Census dataset.

We can explore the possibility in predicting income level based on the individual’s personal information.
it is a good starter example for data pre-processing and machine learning practices.
An individual’s annual income results from various factors. Intuitively, it is influenced by the individual’s education level, age, gender, occupation, and etc.


### About the dataset
Ref: https://www.kaggle.com/jieyima/income-classification-model 

The dataset is based on the US Census bureau. The dataset has many different properties (aka _"X - features"_) like the age, gender, race, country of origin, marital status, education, employment, and the income (the target aka _"Y - label"_) divided into two classes (below or over $50K). For this workshop, this dataset will require little preprocessing before developing our model. 

### About the machine learning model algorithm
Gradient boosting supervised machine learning algorithm with Python is used here to build the model that is capable of predicting the duration of taxi trips in New York city. Python libraries such as Numpy, Pandas, Scikit-learn, XGBoost are used to build the model. 

The machine learning workflow depicted in the diagram below follows the typical supervised machine learning workflow for models development:
- After loading the dataset, the ML algorithm separates data into features (the taxi ride properties) and label (the taxi ride duration). 
- Then the dataset is divided into two parts, one for training the model and one for testing the model. The typical split is 80/20.
- The ML algorithm is then defined and the model is built with the training data set to learn from. 
- Once the model is trained, the ML algorithm tests the model by making predictions on the test data set with the test features. 
- Next, the model accuracy is evaluated by comparing the test predictions to the test labels. Error metrics such as RMSE are used to evaluate the predictions accuracy.
- The trained model is finally saved to a file in the tenant user's working directory in the shared persistent container storage repository. The model is saved to a file so that it can be used in production to serve predictions.
- The next step is then to move your trained model to production to serve your model as a prediction service (Lab Part 4) and make predictions on new data (Lab Part 5).

![ML-Workflow](ML-Workflow.jpg)


Put a section here about the code versioning integrated into the Jupyter Notebook -
Git access
git commands...

### **2- Setup your working directory**

The Python code here is used to create the working directory for your tenant userID. The working directory is used to store the key data components needed in the ML pipeline, such as the input dataset, trained ML model(s) and scoring script(s). 

In [ ]:
userID = "student826"

import os
import shutil
import time

# Shared dataset across all tenant users, use case directory name and scoring file name

datasetFile = "adult_data.csv"
testsetFile = "adult_test.csv"

usecaseDirectory = "UCI_Income"
scoringFile="XGB_Scoring.py"
##scoringFileV2="XGB_Scoringv2.py"

# Project repo path function - file system mount available to all app containers
def ProjectRepo(path):
    ProjectRepo = "/bd-fs-mnt/TenantShare/repo"
    return str(ProjectRepo + '/' + path)


# Delete existing working directory for userID (that may exists from previous execution of the workshop)
studentRepoData = "data" + '/' + usecaseDirectory + '/' + userID
studentRepoModel = "models" + '/' + usecaseDirectory + '/' + userID
studentRepoCode = "code" + '/' + usecaseDirectory + '/' + userID
pathData = ProjectRepo(studentRepoData)
pathModel = ProjectRepo(studentRepoModel)
pathCode = ProjectRepo(studentRepoCode)

if (os.path.exists(ProjectRepo(studentRepoData))):
    #print ("Repository " + pathData + " exists for user " + userID + ". Deleting the repo now...")
    shutil.rmtree(pathData, ignore_errors=True)
    
if (os.path.exists(ProjectRepo(studentRepoModel))):
    #print ("Repository " + pathModel + " exists for user " + userID + ". Deleting the repo now...")
    shutil.rmtree(pathModel, ignore_errors=True)
        
if (os.path.exists(ProjectRepo(studentRepoCode))):
    #print ("Repository " + pathCode + " exists for user " + userID + ". Deleting the repo now...")
    shutil.rmtree(pathCode, ignore_errors=True)
    
# Making sure the input Dataset is loaded and accessible in the shared persistent container storage for your tenant
# Check the dataset file exists in /db-fs-mnt/TenantShare/repo/data/UCI_Income folder:
#print (os.listdir(ProjectRepo('data/UCI_Income')))
datasetFilePath = "data" + '/' + usecaseDirectory + '/' + datasetFile
testsetFilePath = "data" + '/' + usecaseDirectory + '/' + testsetFile
##pathData = ProjectRepo("data" + '/' + usecaseDirectory)

if (not os.path.exists(ProjectRepo(datasetFilePath))):
    print ("Error! Training data set file " + ProjectRepo(datasetFilePath) + " does not exist. Please contact the workshop administrator at hpedev.hackshack@hpe.com before continuing.")
if (not os.path.exists(ProjectRepo(testsetFilePath))):
    print ("Error! Test data set file " + ProjectRepo(testsetFilePath) + " does not exist. Please contact the workshop administrator at hpedev.hackshack@hpe.com before continuing.")
    
print ("Training dataset is " + datasetFilePath)
print ("Test dataset is " + testsetFilePath)

# Define the directory structure for the UserID to store data encoding file, trained model and the scoring script
# /bd-fs-mnt/TenantShare/repo/models/UCI_Income/userID; /bd-fs-mnt/TenantShare/repo/code/UCI_Income/userID

if (not os.path.exists(ProjectRepo(studentRepoData))):
    print ("Creating the Data directory " + pathData)
    try:
        os.makedirs(pathData)
    except OSError:
        print ("Creation of the Data directory %s failed" % pathData)
    else:
        print ("Successfully created the Data directory %s" % pathData)
        
if (not os.path.exists(ProjectRepo(studentRepoModel))):
    print ("Creating the Model directory " + pathModel)
    try:
        os.makedirs(pathModel)
    except OSError:
        print ("Creation of the model directory %s failed" % pathModel)
    else:
        print ("Successfully created the model directory %s" % pathModel)

if (not os.path.exists(ProjectRepo(studentRepoCode))):
    print ("Creating the Code directory " + pathCode)
    try:
        os.makedirs(pathCode)
    except OSError:
        print ("Creation of the code directory %s failed" % pathCode)
    else:
        print ("Successfully created the code directory %s" % pathCode)
        
# Copying scoring script files to your code repository in your working directory
# Make sure the scoring script files are available in the /bd-fs-mnt/TenantShare/repo/code/UCI_Income/userID folder with appropriate permissions (read-execute)
##print ("local directory on your local Jupyter Notebook:")
##print (os.listdir(os.curdir))
##print ("target directory on the shared File System mount for your tenant: ")
##print (os.listdir(pathCode))
srcFile = scoringFile
destFile = pathCode + '/' + scoringFile
##srcFileV2 = scoringFileV2
##destFileV2 = pathCode + '/' + scoringFileV2

if (not os.path.exists(scoringFile)):
    print ("")
    print ("Error! " + scoringFile + " does not exist in your local Jupyter Notebook! Please make sure to copy the file " + scoringFile + " to your local Jupyter Notebook, then re-run that code cell to continue the lab.")
else:    
    if (os.path.exists(pathCode + '/' + scoringFile)):
        print (pathCode + '/' + scoringFile + " file exists. Setting permissions")
        os.chmod (destFile, 0o777)
    else:
        #print ("File" + pathCode + '/' + scoringFile + " does not exist.")
        print ("Copying the scoring script on " + pathCode + " and setting file permissions")
        try:
            shutil.copy(srcFile, destFile)
            os.chmod (destFile, 0o777)
        except IOError as e:
            print ("Unable to copy scoring file. %s" % e )
        except:
            print ("Unexpected error:", sys.exec_info())

##if (not os.path.exists(scoringFileV2)):
##    print (scoringFileV2 + " does not exist in your local Jupyter Notebook! Please make sure to copy the file " + scoringFileV2 + " to your local Jupyter Notebook, then re-run that code cell to continue the lab.")
##else:    
##    if (os.path.exists(pathCode + '/' + scoringFileV2)):
##        print (pathCode + '/' + scoringFileV2 + " file exists. Setting permissions")
##        os.chmod (destFileV2, 0o777)
##    else:
##        print ("File" + pathCode + '/' + scoringFileV2 + " does not exist.")
##        print ("Copying it now on " + pathCode + " and setting file permissions")
##        try:
##            shutil.copy(srcFileV2, destFileV2)
##            os.chmod (destFileV2, 0o777)
##        except IOError as e:
##            print ("Unable to copy scoring file. %s" % e )
##        except:
##            print ("Unexpected error:", sys.exec_info())
            
#print ("target directory  " + pathCode + " :")
#print (os.listdir(pathCode))


### **3- Test Connection to the tenant-shared Training Cluster**

Next, before training your model on remote shared training cluster, you may want to test that the communication with the shared training cluster is indeed functioning properly.

> **Note:** _The Jupyter Notebook ML Ops application used in our solution includes **custom magic** functions to handle remotely submitting training code and retrieving results and logs. The Notebook uses these magic functions to make REST API calls to the API server that runs as part of the shared training environment. These calls submit training jobs and get results from within the Notebook session._

The Jupyter Notebook ML Ops app includes the following **line magic** functions: 
*	%attachments: Returns a list of connected training environments. 
*	%logs --url: URL of the training cluster used to monitor the status of the job submitted to the training cluster.

The Jupyter Notebook ML Ops app also includes the following **cell magic** function:
*	%%training_cluster_name 
This would submit training code to the shared training environment

#### **Get the list of connected training environments**

<b>%attachments</b> is a line magic command that output a table with the name(s) of the training cluster(s) available for us to use. Sometimes, a tenant admin may have created multiple training clusters for different projects depending on the needs of the model or size of data, e.g. some with GPU nodes, while others with CPUs only.

In [ ]:
%attachments

#### **Submit a code cell to a remote training cluster**

To utilize the training cluster, you will need grab the name of the training cluster you want to use and feed it into another custom cell magic command. 

With the **%%trainingengineshared** magic command specified at the beginning of the code cell, the Jupyter Notebook will 
submit the entire content of the cell to the training cluster named _trainingengineshared_. If you comment this magic command, the code will run on your local Jupyter Notebook.

The example cell below will execute a print statement on the training cluster.

In [ ]:
%%trainingengineshared

import datetime

print('test')
print("Date time: ", datetime.datetime.now())

#### **Retrieve the result of the job**

The training cluster will send back a unique log url for the job submitted.   
You can use the _History URL_ with the _"%log --url"_ custom **line magic** command to track the status of the job in real time. 

Copy the History URL from the output of the previous cell and paste it into the cell below where it says "your_http_url_here", and run the cell code.

A status of "**Finished**" means the execution of the job submitted to the training cluster has completed.

In [ ]:
##%logs --url your_http_url_here
%logs --url your_http_url_here

### **4- Data preprocessing**

The data is prepared for ML task. Preparing the data involves not only collecting the data but then data cleaning, and transforming that data into a format that can be utilized by the learning algorithm. This involves many tasks klike dealing with missing values or malformed data.

In [ ]:
import numpy as np
import pandas as pd
import os
import json
import datetime
import seaborn as sns
sns.set(font_scale=1.5)

%matplotlib inline 

In [ ]:
# Project repo path function - file system mount available to all app containers
def ProjectRepo(path):
    ProjectRepo = "/bd-fs-mnt/TenantShare/repo"
    return str(ProjectRepo + '/' + path)

In [ ]:
datasetFile = "adult_data.csv"
testsetFile = "adult_test.csv"
usecaseDirectory = "UCI_Income"

#strpath= "data/UCI_Income/adult_data.csv"
strpath= "data" + '/' + usecaseDirectory + '/' + datasetFile
train_file = ProjectRepo(strpath)
train_set = pd.read_csv(train_file, header=None)
train_set.head()

In [ ]:
##strpath= "data/UCI_Income/adult_test.csv"
strpath= "data" + '/' + usecaseDirectory + '/' + testsetFile
test_file = ProjectRepo(strpath)
test_set = pd.read_csv(test_file, skiprows=1, header=None)
test_set.head()

#### **Initial Findings**
1. No column headers in the data (can fix using dataset description from website)
2. Some "?" in test data 
3. Target values differ in train and test set

#### 1. Fix column headers

In [ ]:
col_labels = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 
              'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
             'wage_class']
train_set.columns = col_labels
test_set.columns = col_labels
train_set.info()
test_set.info()

#### 2. Clean up ? in data

In [ ]:
train_set.replace(' ?', np.nan).dropna().shape
test_set.replace(' ?', np.nan).dropna().shape
# removing rows with "?" from our dataframes 
train_no_missing = train_set.replace(' ?', np.nan).dropna()
test_no_missing = test_set.replace(' ?', np.nan).dropna()

#### 3. Fix targets (remove the extra periods from '<=50K.' to '<=50K')

In [ ]:
test_no_missing['wage_class'] = test_no_missing.wage_class.replace({' <=50K.' : ' <=50K', ' >50K.' : ' >50K'})
test_no_missing.wage_class.unique()
train_no_missing.wage_class.unique()

#### **Applying ordinal encoding to categoricals**

Machine learning models require all input and output variables to be numeric. Therefore categorical data (non numeric data) must be transformed (encoded) to numbers before we can use the data. Encoding is a required preprocessing step when working with categorical data for machine learning algorithms.

- Here we use ordinal encoding technique (for categorical variables that have a natural rank ordering): convert string labels to integer values 1 through k. First unique value in column becomes 1, the second becomes 2, the third becomes 3, and so on


In [ ]:
#combine the datasets together first
combined_set = pd.concat([train_no_missing, test_no_missing], axis=0)
combined_set.info()
#Visualizations after initial cleaning of dataset 
group = combined_set.groupby('wage_class')
group
#encode non-numerical features into numeric values using pandas Cateogrical codes 
#and generating categorical codes mapping into dictionary
cat_codes = {}
for feature in combined_set.columns: 
    if combined_set[feature].dtype == 'object':
        #workclass : { occupation : number }
        temp_dict = {}
        feature_codes = list(pd.Categorical(combined_set[feature]).codes)
        feature_list = list(combined_set[feature])
        for i in range(len(feature_codes)):
            temp_dict[feature_list[i].strip()] = int(feature_codes[i])
            if len(temp_dict) > len(feature_list):
                break
        cat_codes[feature] = temp_dict
        combined_set[feature] = pd.Categorical(combined_set[feature]).codes
combined_set.info()
# saving encoding to json file to be used for scoring script
strpath= "data/UCI_Income/" + userID + "/encoding.json"
json_file = ProjectRepo(strpath)
with open(json_file, 'w') as file:
    json.dump(cat_codes, file)
    #split combined set back into test/train split 
final_train = combined_set[:train_no_missing.shape[0]] 
final_test = combined_set[train_no_missing.shape[0]:]
strpath= "data/UCI_Income/" + userID + "/adult_train_cleaned.csv"
final_train.to_csv(ProjectRepo(strpath))
strpath= "data/UCI_Income/" + userID + "/adult_test_cleaned.csv"
final_test.to_csv(ProjectRepo(strpath))
#extracting target values from our test and train sets 
y_train = final_train.pop('wage_class')
y_test = final_test.pop('wage_class')

### **5- Model development**

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

#### **First model**

In [ ]:
cv_params = {'max_depth': [3,5,7], 'min_child_weight': [1,3,5]}
ind_params = {'learning_Rate': 0.1, 'n_estimators': 1000, 'seed': 0, 'subsample' : 0.8, 'colsample_bytree': 0.8, 
              'objective': 'binary:logistic'}

#optimizing for accuracy, GBM = gradient boost model
optimized_GBM = GridSearchCV(xgb.XGBClassifier(**ind_params), 
                             cv_params, 
                             scoring = 'accuracy', cv = 5, n_jobs = -1)

In [ ]:
print("Start time - training fit: ", datetime.datetime.now())
optimized_GBM.fit(final_train, y_train)
print("End time - training fit: ", datetime.datetime.now())

In [ ]:
optimized_GBM.cv_results_

#### **Second model**
Tuning other hyperparameters in an attempt to achieve higher mean accuracy

In [ ]:
cv_params = {'learning_rate': [0.1, 0.01], 'subsample': [0.7, 0.8, 0.9]}
ind_params = {'n_estimators': 1000, 'seed': 0, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 
              'max_depth': 3, 'min_child_weight': 1}
                    
optimized_GBM = GridSearchCV(xgb.XGBClassifier(**ind_params), 
                             cv_params, 
                             scoring = 'accuracy', cv=5, n_jobs=-1)
print("Start time - training fit: ", datetime.datetime.now())
optimized_GBM.fit(final_train, y_train)
print("End time - training fit: ", datetime.datetime.now())

In [ ]:
optimized_GBM.cv_results_

#### **Third model**
Utilize XGBoost's built-in cv which allows early stopping to prevent overfitting

In [ ]:
xgdmat = xgb.DMatrix(final_train, y_train)

In [ ]:
print("Start time - training fit: ", datetime.datetime.now())
our_params = {'eta': 0.1, 'seed': 0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic',
              'max_depth': 3, 'min_child_weight': 1}

cv_xgb = xgb.cv(params=our_params, dtrain=xgdmat, num_boost_round=3000, metrics=['error'],
                early_stopping_rounds=100)
print("End time - training fit: ", datetime.datetime.now())

In [ ]:
print('Best iteration:', len(cv_xgb))

In [ ]:
cv_xgb.tail(5)

#### **Final Model**

In [ ]:
print("Start time - training fit: ", datetime.datetime.now())
our_params = {'eta': 0.1, 'seed':0, 'subsample': 0.8, 'colsample_bytree': 0.8, 
             'objective': 'binary:logistic', 'max_depth':3, 'min_child_weight':1} 

final_gb = xgb.train(our_params, xgdmat, num_boost_round = 326)
print("End time - training fit: ", datetime.datetime.now())

#### **Plot feature importances**

In [ ]:
xgb.plot_importance(final_gb)

In [ ]:
importances = final_gb.get_fscore()
importances

In [ ]:
importance_frame = pd.DataFrame({'Importance': list(importances.values()), 'Feature': list(importances.keys())})
importance_frame.sort_values(by = 'Importance', inplace=True)
importance_frame.plot(kind='barh', x='Feature', figsize=(8,8), color='green')

### **6- Train model remotely on a large capacity training cluster**

After getting done with the model development and tuning part, data scientists typically submit the code they developed in their local Jupyter Notebook to a remote larger capacity training cluster to train their models in a reasonable time against a larger dataset.

In this part of the lab, to help illustrate this step, you will submit the code to a remote training cluster environment to train your models in a reasonable time. This training job combines all the cells we've worked on previously and form one large cell. At the end, we will save the model into the Project Repository. 

At the end of the execution of the code cell, the model is saved to a file in the central project repository (i.e.: your tenant user's working directory) for later use in production deployment engine.

In [ ]:
%%trainingengineshared

userID="student826"

# Importing libraries 
print("Importing libraries")
import numpy as np
import pandas as pd
import os
import pickle
import xgboost as xgb
import datetime
from sklearn.model_selection import GridSearchCV

# Start time 
print("Start time: ", datetime.datetime.now())

# Project repo path function
##def ProjectRepo(path):
##   ProjectRepo = os.popen('bdvcli --get cluster.project_repo').read().rstrip()
##   return ProjectRepo + '/' + path

# Project repo path function - file system mount available to all app containers
def ProjectRepo(path):
    ProjectRepo = "/bd-fs-mnt/TenantShare/repo"
    return str(ProjectRepo + '/' + path)

# Reading in data 
print("Reading in data")
strpath= "data/UCI_Income/" + userID + "/adult_train_cleaned.csv"
train = pd.read_csv(ProjectRepo(strpath))
print("Done reading in data")

# Extracting target values 
y_train = train.pop('wage_class')
train.pop('Unnamed: 0')

# Model development / Training
print("Training...")
print("Start time: ", datetime.datetime.now())
xgdmat = xgb.DMatrix(train, y_train)
our_params = {'eta': 0.1, 'seed': 0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic',
              'max_depth': 3, 'min_child_weight': 1}
cv_xgb = xgb.cv(params=our_params, dtrain=xgdmat, num_boost_round=3000, metrics=['error'],
                early_stopping_rounds=100)
optimal_rounds = len(cv_xgb)
final_gb = xgb.train(our_params, xgdmat, num_boost_round = optimal_rounds)

print("Training end time: ", datetime.datetime.now())

# Save model into project repo
print("Saving model")
strpath= "models/UCI_Income/" + userID + "/XGB.pickle.dat"
print("Saving model in a pickle file as " + ProjectRepo(strpath))
xgb.Booster.save_model(final_gb, ProjectRepo(strpath))

# Finish time
print("End time: ", datetime.datetime.now())

#### **Monitor the training job**

##### Copy the unique log _History URL_ above and paste it into the cell below to monitor your training job (run the cell code below regularly until the job is marked as "**finished**")

>**Note:** Depending on the number of concurrent jobs submitted to the training cluster environment (i.e: multiple participants run the workshop concurrently), the model training may take several minutes to complete.
Copy the unique log url and paste it into the cell below.

In [ ]:
##%logs --url your_http_url_here
%logs --url your_http_url_here

### **7-Testing Models**
Here we are going to test that the model prediction is as expected

1. We're going to test the model that we've created here locally 
2. Then we will test the model that has been saved in the Project Repository 
3. Validate that the values are the same

We will take the first value in the adult_test_cleaned dataset
Test the model by loading from Project Repository

In [ ]:
strpath= "data/UCI_Income/" + userID + "/adult_test_cleaned.csv"
cleaned = pd.read_csv(ProjectRepo(strpath))
cleaned.tail(1)

In [ ]:
#Running with final_gb model from local notebook
temp = cleaned.tail(1)
y_test = temp.pop('wage_class')
temp.set_index('age')
temp.pop('Unnamed: 0')
mat = xgb.DMatrix(temp) 
y_pred = final_gb.predict(mat)
y_pred

In [ ]:
model = xgb.Booster({'nthread':430})
strpath= "models/UCI_Income/" + userID + "/XGB.pickle.dat"
model.load_model(ProjectRepo(strpath))
temp = cleaned.tail(1)
y_test = temp.pop('wage_class')
temp.set_index('age')
temp.pop('Unnamed: 0')
mat = xgb.DMatrix(temp) 
y_pred = model.predict(mat)
y_pred

Validate that the 2 numbers are the same 

### **8- Model registry and deployment**

##### After the model development and training, you are now ready to deploy your trained model as prediction service and start serving queries.

<font color="red">**Now, let's go back to your JupyterHub account session to continue the hands-on from Lab 4 for model registry and model deployment:**</font>
<font color="blue">**4-WKSHP-MLOps-K8s-Register-Model-Deployment.ipynb**.</font>